# L9b: Text Embeddings: Bag-of-Words, TF-IDF, and PMI
In this lab, we explore static text representations by building a vocabulary, constructing a bag-of-words (BoW) matrix, computing TF-IDF weights, and estimating pointwise mutual information (PMI). We also train a continuous bag-of-words (CBOW) model and examine the word embeddings it learns.

> __Learning Objectives__
> * __Build BoW and vocabulary representations:__ Construct a vocabulary from a toy corpus and encode documents as count vectors, then discuss the limitations of this representation.
> * __Compute TF-IDF and compare documents:__ Apply term frequency-inverse document frequency weighting and use cosine similarity to measure document-level similarity.
> * __Estimate PMI and train CBOW embeddings:__ Compute pointwise mutual information to measure word-word association strength, then train a CBOW model and extract dense word embeddings.

___

## Setup, Data, and Prerequisites

> We load packages and source files via `Include.jl`. This file loads `VLDataScienceMachineLearningPackage.jl` along with several local source files that implement the BoW, TF-IDF, PMI, and CBOW helper functions used in this lab.

In [ ]:
include("Include.jl")

### Implementations
The following functions are used in this lab. They are defined in the `src/` directory and loaded by `Include.jl`.

| Function | Location | Description |
|----------|----------|-------------|
| `build_bow_matrix(sentences, vocabulary)` | `src/BagOfWords.jl` | Builds a count matrix of shape `(num_sentences × vocab_size)` |
| `hashing_vectorizer(words; length)` | `src/BagOfWords.jl` | Maps words to a fixed-length vector using the hashing trick |
| `build_tf_matrix(bow_matrix)` | `src/TFIDF.jl` | Normalizes BoW counts by sentence length to get term frequency |
| `build_idf_dictionary(bow_matrix, vocabulary, N)` | `src/TFIDF.jl` | Computes smoothed inverse document frequency for each term |
| `build_tfidf_matrix(tf_matrix, idf_dict, inv_vocab)` | `src/TFIDF.jl` | Combines TF and IDF to produce TF-IDF weight matrix |
| `build_cooccurrence_matrix(sentences, vocabulary; window_size)` | `src/PMI.jl` | Counts word co-occurrences within a sliding window |
| `build_pmi_matrices(cooccurrence_matrix)` | `src/PMI.jl` | Returns PMI and PPMI matrices from a co-occurrence matrix |
| `build_cbow_pairs(sentences, vocabulary; window_size)` | `src/CBOW.jl` | Generates (context, target) training pairs for CBOW |
| `train_cbow(training_pairs, vocab_size; d_h, eta, num_epochs)` | `src/CBOW.jl` | Trains a CBOW model via backpropagation, returns `W1, W2, loss_history` |

### Constants
We define the constants used throughout this lab.

In [ ]:
# corpus
sentences = [
    "I love julia and machine learning",
    "julia is fun and great",
    "I enjoy data science and machine learning",
    "data science is great and fun",
    "machine learning is a great and fun subject",
    "I love learning new things about data science"
];

# control tokens
control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"];

# CBOW hyperparameters
window_size = 2;   # context window half-width
d_h = 5;           # embedding (hidden layer) dimension
eta = 0.01;        # learning rate
num_epochs = 500;  # training epochs

### Data
We build a vocabulary from the corpus. All tokens are converted to lowercase. Four control tokens (`<bos>`, `<eos>`, `<pad>`, `<unk>`) are added at the start of the vocabulary.

In [ ]:
# build vocabulary: assign integer index to each unique word
all_words = vcat([split(lowercase(s)) for s in sentences]...)
unique_words = unique(all_words)
vocabulary = Dict{String, Int64}()
for (i, token) in enumerate(control_tokens)
    vocabulary[token] = i
end
offset = length(control_tokens)
for (i, word) in enumerate(unique_words)
    if !haskey(vocabulary, word)
        vocabulary[word] = offset + i
    end
end
inverse_vocabulary = Dict{Int64, String}(v => k for (k, v) in vocabulary)
println("Vocabulary size: $(length(vocabulary))")

___

## Task 1: Build the Bag-of-Words Matrix

> In a bag-of-words (BoW) representation, each document is encoded as a vector of word counts over the vocabulary. The representation discards word order and treats each document as an unordered collection of terms. We call `build_bow_matrix`, which augments each sentence with `<bos>` and `<eos>` tokens before counting.

In [ ]:
bow_matrix = build_bow_matrix(sentences, vocabulary);
println("BoW matrix size: $(size(bow_matrix))  (sentences × vocabulary)")
bow_matrix

We can also use the hashing trick to map words to a fixed-length vector without an explicit vocabulary.

In [ ]:
# hashing vectorizer for sentence 1
words_s1 = split(lowercase(sentences[1]))
hashed_vec = hashing_vectorizer(words_s1; length=16)
println("Hashing vectorizer output (length 16): ", hashed_vec)

__Discussion:__ The hashing vectorizer avoids building an explicit vocabulary. What are the trade-offs? Under what conditions would hash collisions be most problematic?

___

## Task 2: Compute TF-IDF Weights and Document Similarity

> Term Frequency-Inverse Document Frequency (TF-IDF) re-weights BoW counts so that frequent words across all documents (e.g., "and", "is") receive low weight while words that are distinctive to a few documents receive high weight. TF for term $t$ in document $d$ is $\text{TF}(t,d) = \frac{n_{t,d}}{\sum_{t'} n_{t',d}}$ and the smoothed IDF is $\text{IDF}(t) = \ln\!\left(\frac{N+1}{df_t+1}\right)$, where $N$ is the number of documents and $df_t$ is the document frequency of term $t$.

In [ ]:
TF_matrix = build_tf_matrix(bow_matrix);
IDF_dict = build_idf_dictionary(bow_matrix, vocabulary, length(sentences));
TFIDF_matrix = build_tfidf_matrix(TF_matrix, IDF_dict, inverse_vocabulary);
println("TF-IDF matrix size: $(size(TFIDF_matrix))");

We compute pairwise cosine similarity between documents using their TF-IDF vectors.

In [ ]:
n = length(sentences)
cosine_sim = zeros(n, n)
for i in 1:n
    for j in 1:n
        u = TFIDF_matrix[i, :]
        v = TFIDF_matrix[j, :]
        denom = norm(u) * norm(v)
        cosine_sim[i,j] = denom > 0 ? dot(u, v) / denom : 0.0
    end
end

# display as a table
header = ["S$i" for i in 1:n]
data = round.(cosine_sim, digits=3)
pretty_table(data; header=header, row_labels=["S$i" for i in 1:n])

__Discussion:__ Which sentence pairs have the highest cosine similarity? Do the high-similarity pairs share distinctive words, or only common function words? What does a cosine similarity of 1.0 indicate?

___

## Task 3: Estimate PMI and Analyze Word Associations

> Pointwise Mutual Information (PMI) measures how much more often a word pair $(w, c)$ co-occurs than expected under independence: $\text{PMI}(w,c) = \log_2\!\frac{P(w,c)}{P(w)\,P(c)}$. Positive values indicate that $w$ and $c$ co-occur more than chance; negative values indicate the opposite. Positive PMI (PPMI) clips negative values to zero and is more commonly used in practice.

In [ ]:
cooccurrence_matrix = build_cooccurrence_matrix(sentences, vocabulary; window_size=window_size);
PMI_matrix, PPMI_matrix = build_pmi_matrices(cooccurrence_matrix);
println("Co-occurrence matrix size: $(size(cooccurrence_matrix))")

We inspect the PPMI values for a set of word pairs.

In [ ]:
word_pairs = [
    ("machine", "learning"),
    ("data", "science"),
    ("julia", "fun"),
    ("love", "julia"),
    ("great", "fun"),
    ("machine", "julia"),
]

header_ppmi = ["Word 1", "Word 2", "Co-occurrence", "PPMI"]
rows = []
for (w1, w2) in word_pairs
    i = get(vocabulary, w1, 0)
    j = get(vocabulary, w2, 0)
    if i > 0 && j > 0
        co = cooccurrence_matrix[i, j]
        ppmi_val = isfinite(PPMI_matrix[i, j]) ? round(PPMI_matrix[i,j], digits=3) : 0.0
        push!(rows, [w1, w2, co, ppmi_val])
    end
end
pretty_table(hcat(rows...)'; header=header_ppmi)

__Discussion:__ The pair ("love", "julia") has zero co-occurrence even though both words appear in sentence 1. Explain why. How would increasing `window_size` change this result? What does a PPMI of 0 indicate versus a missing entry?

___

## Task 4: Train CBOW and Extract Word Embeddings

> The Continuous Bag-of-Words (CBOW) model learns dense word embeddings by training a shallow neural network to predict a target word from the sum of its context word one-hot vectors. The network has a single hidden layer (no activation function) and a softmax output. After training, the columns of the input weight matrix $\mathbf{W}_1 \in \mathbb{R}^{d_h \times |V|}$ are the word embeddings.

In [ ]:
training_pairs = build_cbow_pairs(sentences, vocabulary; window_size=window_size);
println("Number of training pairs: $(length(training_pairs))")

In [ ]:
Random.seed!(42)
W1, W2, loss_history = train_cbow(training_pairs, length(vocabulary); 
    d_h=d_h, eta=eta, num_epochs=num_epochs);

plot(loss_history; xlabel="Epoch", ylabel="Average Cross-Entropy Loss",
    title="CBOW Training Loss", legend=false, lw=2)

We compare the cosine similarity between word pairs using the learned embeddings.

In [ ]:
word_pairs_emb = [
    ("machine", "learning"),
    ("data", "science"),
    ("julia", "fun"),
    ("love", "enjoy"),
    ("great", "fun"),
    ("machine", "julia"),
]

header_emb = ["Word 1", "Word 2", "Cosine Similarity"]
rows_emb = []
for (w1, w2) in word_pairs_emb
    i = get(vocabulary, w1, 0)
    j = get(vocabulary, w2, 0)
    if i > 0 && j > 0
        e1 = W1[:, i]
        e2 = W1[:, j]
        sim = dot(e1, e2) / (norm(e1) * norm(e2) + 1e-10)
        push!(rows_emb, [w1, w2, round(sim, digits=3)])
    end
end
pretty_table(hcat(rows_emb...)'; header=header_emb)

__Discussion:__ Compare the PPMI values from Task 3 to the CBOW cosine similarities. Do both methods agree on which pairs are most associated? What can CBOW capture that PPMI cannot?

___

## Summary

In this lab, we built static and learned text representations for a toy corpus. We progressed from sparse count vectors (BoW) through re-weighted representations (TF-IDF) and association statistics (PMI/PPMI) to dense learned embeddings (CBOW).

> __Key Takeaways__
> * **Bag-of-Words and sparsity:** BoW encodes documents as count vectors over a vocabulary, but the resulting representations are high-dimensional, sparse, and lose word order and semantic context.
> * **TF-IDF down-weights common terms:** TF-IDF re-weights BoW counts so that terms distinctive to a small number of documents receive higher weight, making document-level cosine similarity more meaningful.
> * **PMI and CBOW capture word associations differently:** PMI measures association from co-occurrence counts directly, while CBOW learns dense embeddings through prediction. Both can surface semantically related pairs, but CBOW generalizes beyond the training window.